# SYNTHIA → Cityscapes Pipeline (End-to-End)

DAMP prompt training → CAM generation → Evaluation → CRF pseudo-masks

| Step | Description |
|------|-------------|
| 0 | Setup: clone repo, install deps, download data |
| 1 | Prepare SYNTHIA labels (16-bit → Cityscapes train IDs) |
| 2 | Download & prepare Cityscapes |
| 3 | Build multilabel JSON |
| 4 | Train DAMP (fast config: batch 64, 30 epochs) |
| 5 | Generate CAMs on SYNTHIA |
| 6 | Evaluate CAMs |
| 7 | CRF post-processing → pseudo-masks |

In [ ]:
# ===== CELL 0a: MOUNT DRIVE =====
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ===== CELL 0b: CLONE REPO + INSTALL DEPS =====
import os, sys

# Clone if not already
REPO_DIR = "/content/Damp_es"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/baominh5xx2/Damp_es_CS338.git {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

# Install dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q timm yacs ftfy regex pydensecrf lxml ttach
!pip install -q opencv-python-headless scikit-learn matplotlib tqdm
!pip install -q datasets huggingface_hub pyarrow
!pip install -q git+https://github.com/KaiyangZhou/Dassl.pytorch.git

# Verify
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ===== CELL 0c: CONFIG — paths & settings =====
from pathlib import Path

# ── Change this to your Drive path ──
DATA_ROOT = Path("/content/drive/MyDrive/datasets/synthia_cs338")
OUTPUT_DIR = Path("/content/drive/MyDrive/datasets/synthia_cs338/output")

# Derived paths
SYNTHIA_RAW = DATA_ROOT / "data" / "raw" / "synthia"
CITY_RAW    = DATA_ROOT / "data" / "raw" / "cityscapes"
PROCESSED   = DATA_ROOT / "data" / "processed"
DAMP_DIR    = OUTPUT_DIR / "damp" / "synthia"
CAM_DIR     = OUTPUT_DIR / "synthia" / "cams_damp"
MASK_DIR    = OUTPUT_DIR / "synthia" / "pseudo_masks"

SYNTHIA_IMG   = SYNTHIA_RAW / "images"
SYNTHIA_LBL   = SYNTHIA_RAW / "labels"
SYNTHIA_SPLIT = SYNTHIA_RAW / "splits" / "train.txt"
CITY_IMG      = CITY_RAW / "images"
CITY_LBL      = CITY_RAW / "labels"
CITY_TRAIN_SPLIT = CITY_RAW / "splits" / "train.txt"
CITY_VAL_SPLIT   = CITY_RAW / "splits" / "val.txt"

HF_SYNTHIA_REPO = "Minhbao5xx2/synthia-rand-cityscapes-16class-parquet_fix"
HF_CITY_REPO    = "Chris1/cityscapes"

print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"HF SYNTHIA : {HF_SYNTHIA_REPO}")

## Step 1: Download & Prepare SYNTHIA

Downloads from HuggingFace (fixed parquet with correct 16-bit labels), then converts to images/labels/splits.

In [ ]:
# ===== CELL 1: DOWNLOAD + PREPARE SYNTHIA =====
import os, glob
from pathlib import Path

PARQUET_DIR = DATA_ROOT / "synthia_parquet"

# ── 1a: Download parquet from HuggingFace ──
if not SYNTHIA_SPLIT.exists():
    print("Downloading SYNTHIA parquet from HuggingFace ...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=HF_SYNTHIA_REPO,
        repo_type="dataset",
        local_dir=str(PARQUET_DIR),
        max_workers=8,
    )
    print("Download complete.")
else:
    print("SYNTHIA data already prepared, skipping download.")

# ── 1b: Convert parquet → images/labels/splits ──
if not SYNTHIA_SPLIT.exists():
    print("Converting parquet to images/labels/splits ...")
    !python {REPO_DIR}/tools/prepare_synthia_hf.py \
        --parquet-dir {PARQUET_DIR / "parquet"} \
        --output-root {SYNTHIA_RAW} \
        --num-workers 4
else:
    print("SYNTHIA splits already exist, skipping conversion.")

# ── 1c: Verify ──
n_img = len(list(Path(SYNTHIA_IMG).glob("*.png"))) if SYNTHIA_IMG.exists() else 0
n_lbl = len(list(Path(SYNTHIA_LBL).glob("*.png"))) if SYNTHIA_LBL.exists() else 0
n_split = len(open(SYNTHIA_SPLIT).readlines()) if SYNTHIA_SPLIT.exists() else 0
print(f"\nSYNTHIA: {n_img} images, {n_lbl} labels, {n_split} split entries")

# Quick label sanity check
if n_lbl > 0:
    import cv2, numpy as np
    sample_lbl = sorted(Path(SYNTHIA_LBL).glob("*.png"))[0]
    arr = cv2.imread(str(sample_lbl), cv2.IMREAD_UNCHANGED)
    if arr.ndim == 3:
        unique = np.unique(arr[:,:,2])  # Red channel = class ID (16-bit)
    else:
        unique = np.unique(arr)
    n_classes = len([v for v in unique if v != 0])
    print(f"Label check ({sample_lbl.name}): {n_classes} valid classes, unique IDs: {unique.tolist()[:15]}")

## Step 2: Download & Prepare Cityscapes

In [ ]:
# ===== CELL 2: DOWNLOAD + PREPARE CITYSCAPES =====
if not CITY_VAL_SPLIT.exists():
    print("Downloading & preparing Cityscapes from HuggingFace ...")
    !python {REPO_DIR}/tools/prepare_cityscapes_hf.py \
        --dataset-id {HF_CITY_REPO} \
        --output-root {CITY_RAW} \
        --splits train,validation \
        --num-workers 8
else:
    print("Cityscapes already prepared.")

# Verify
n_city_img = len(list(Path(CITY_IMG).glob("*.png"))) if CITY_IMG.exists() else 0
n_city_train = len(open(CITY_TRAIN_SPLIT).readlines()) if CITY_TRAIN_SPLIT.exists() else 0
n_city_val = len(open(CITY_VAL_SPLIT).readlines()) if CITY_VAL_SPLIT.exists() else 0
print(f"Cityscapes: {n_city_img} images, {n_city_train} train, {n_city_val} val")

## Step 3: Build Multilabel JSON

Extracts per-image class labels from segmentation masks for multi-label classification.

In [ ]:
# ===== CELL 3: BUILD MULTILABEL JSON =====
SYNTHIA_ML_DIR = PROCESSED / "synthia_multilabel"
CITY_ML_DIR    = PROCESSED / "cityscapes_multilabel"

# ── SYNTHIA multilabel ──
synthia_ml_file = SYNTHIA_ML_DIR / "multilabel.json"
if not synthia_ml_file.exists():
    print("Building SYNTHIA multilabel ...")
    !python {REPO_DIR}/tools/build_synthia_multilabel.py \
        --split-file {SYNTHIA_SPLIT} \
        --label-dir {SYNTHIA_LBL} \
        --output-dir {SYNTHIA_ML_DIR} \
        --num-workers 8
else:
    print("SYNTHIA multilabel already exists.")

# ── Cityscapes train multilabel ──
city_train_ml = CITY_ML_DIR / "train_multilabel.json"
if not city_train_ml.exists():
    print("Building Cityscapes train multilabel ...")
    !python {REPO_DIR}/tools/build_cityscapes_multilabel.py \
        --split-file {CITY_TRAIN_SPLIT} \
        --label-dir {CITY_LBL} \
        --output-dir {CITY_ML_DIR} \
        --output-file train_multilabel.json \
        --num-workers 8
else:
    print("Cityscapes train multilabel already exists.")

# ── Cityscapes val multilabel ──
city_val_ml = CITY_ML_DIR / "val_multilabel.json"
if not city_val_ml.exists():
    print("Building Cityscapes val multilabel ...")
    !python {REPO_DIR}/tools/build_cityscapes_multilabel.py \
        --split-file {CITY_VAL_SPLIT} \
        --label-dir {CITY_LBL} \
        --output-dir {CITY_ML_DIR} \
        --output-file val_multilabel.json \
        --num-workers 8
else:
    print("Cityscapes val multilabel already exists.")

print("\nAll multilabel files ready!")

## Step 4: Train DAMP Prompts (Fast Config)

Batch size 64, 30 epochs, LR 0.008, 8 workers, `torch.compile()`.

This is the most time-consuming step (~15-30 min on T4, ~5-10 min on A100).

In [ ]:
# ===== CELL 4: TRAIN DAMP =====
%cd {REPO_DIR}

PROMPT_CKPT = DAMP_DIR / "prompt_learner.pth"

if PROMPT_CKPT.exists():
    print(f"DAMP checkpoint already exists: {PROMPT_CKPT}")
    print("Delete it to retrain.")
else:
    print(f"Training DAMP — output to {DAMP_DIR}")
    !python train.py \
        --config-file configs/trainers/damp_synthia_fast.yaml \
        DATASET.ROOT {DATA_ROOT} \
        OUTPUT_DIR {DAMP_DIR}

# Verify
if PROMPT_CKPT.exists():
    import os
    size_mb = os.path.getsize(PROMPT_CKPT) / 1024 / 1024
    print(f"\nPrompt checkpoint: {PROMPT_CKPT} ({size_mb:.1f} MB)")
else:
    print("\nERROR: prompt_learner.pth not found! Training may have failed.")

## Step 5: Generate CAMs on SYNTHIA

Uses DAMP prompts + CLIP attention to generate class activation maps.

In [ ]:
# ===== CELL 5: GENERATE CAMs =====
%cd {REPO_DIR}

import glob
n_existing = len(glob.glob(str(CAM_DIR / "*.npy"))) if CAM_DIR.exists() else 0

if n_existing >= 9400:
    print(f"CAMs already generated ({n_existing} files). Skipping.")
else:
    print(f"Generating CAMs → {CAM_DIR}")
    !python generate_cams.py \
        --dataset synthia \
        --img_root {SYNTHIA_IMG} \
        --label_root {SYNTHIA_LBL} \
        --split_file {SYNTHIA_SPLIT} \
        --cam_out_dir {CAM_DIR} \
        --damp_prompt_ckpt {PROMPT_CKPT} \
        --max_long_side 1024 \
        --skip_existing

# Verify
n_cam = len(glob.glob(str(CAM_DIR / "*.npy")))
print(f"\n{ n_cam} .npy CAM files in {CAM_DIR}")
if n_cam == 0:
    print("ERROR: No CAM files generated!")

## Step 6: Evaluate CAMs

Threshold grid search to find best mIoU.

In [ ]:
# ===== CELL 6: EVALUATE CAMs =====
%cd {REPO_DIR}

print("Evaluating CAMs (threshold grid search) ...")
!python eval_cam.py \
    --dataset synthia \
    --cam_out_dir {CAM_DIR} \
    --gt_root {SYNTHIA_LBL} \
    --split_file {SYNTHIA_SPLIT} \
    --cam_type attn_highres

## Step 7: CRF Post-processing → Pseudo-masks

In [ ]:
# ===== CELL 7: CRF PSEUDO-MASKS =====
%cd {REPO_DIR}

print("CRF post-processing → pseudo-masks ...")
!python eval_cam.py \
    --dataset synthia \
    --cam_out_dir {CAM_DIR} \
    --gt_root {SYNTHIA_LBL} \
    --split_file {SYNTHIA_SPLIT} \
    --cam_type attn_highres \
    --crf \
    --image_root {SYNTHIA_IMG} \
    --mask_output_dir {MASK_DIR}

import glob
n_masks = len(glob.glob(str(MASK_DIR / "*.png")))
print(f"\n{n_masks} pseudo-masks saved to {MASK_DIR}")
print("\n============================================")
print(" SYNTHIA Pipeline complete!")
print(f" Pseudo-masks: {MASK_DIR}")
print("============================================")